In [ ]:
import kagglehub
import pandas as pd
import os
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings('ignore')
# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
Q1_data_path = os.path.join(path, 'Q1_data.csv')
df_Q1_data = pd.read_csv(Q1_data_path)
print(f"Shape: {df_Q1_data.shape}")
df_Q1_data.head()

In [ ]:
# Task 2: Write your code here:
df_Q1_data.head()

In [ ]:
# Task 3: Write your code here:
df_Q1_data.info()

In [ ]:
# Task 4: Write your code here:
df_Q1_data.describe()

In [ ]:
# Task 5: Write your code here:
print(f"Fast: {df_Q1_data['Delivery_Time'].sum()}")
print(f"Normal: {(df_Q1_data['Delivery_Time'] == 0).sum()}")

In [ ]:
# Task 1: Write your code here:
df_Q1_data = df_Q1_data.drop('Order_ID', axis=1)


In [ ]:
# Task 2: Write your code here:
for col in ['Weather', 'Traffic_Level', 'Time_of_Day']:
    mode_val = df_Q1_data[col].mode()[0]
    df_Q1_data[col] = df_Q1_data[col].fillna(mode_val)

for col in ['Courier_Experience_yrs', 'Delivery_Time']:
    median_val = df_Q1_data[col].median()
    df_Q1_data[col] = df_Q1_data[col].fillna(median_val)


In [ ]:
# Task 3: Write your code here:
initial_rows = df_Q1_data.shape[0]
df_Q1_data.drop_duplicates(inplace=True)
duplicate_rows = initial_rows - df_Q1_data.shape[0]
print(f"Number of duplicate rows removed: {duplicate_rows}")


In [ ]:
# Task 4: Write your code here:
categorical_cols = ['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']
df_Q1_data_processed = pd.get_dummies(df_Q1_data, columns=categorical_cols, drop_first=True)


In [ ]:
# Task 5: Write your code here:
scaler = StandardScaler()

X = df_Q1_data_processed.drop('Delivery_Time', axis=1)
y = df_Q1_data_processed['Delivery_Time']

X_scaled = scaler.fit_transform(X)
X = pd.DataFrame(X_scaled, columns=X.columns)


In [ ]:
# Task 6: Write your code here:
print("Delivery_Time is a continuous numerical target, so checking for 'imbalance' in the classification sense is not applicable.")
print(y.describe())
plt.figure(figsize=(10, 6))
sns.histplot(y, kde=True)
plt.title('Distribution of Delivery Time')
plt.xlabel('Delivery Time (minutes)')
plt.ylabel('Frequency')
plt.show()


In [ ]:
# Task 1: X and y are already defined and prepared from the previous steps.
# X contains the scaled features and y contains the 'Delivery_Time' target.

In [ ]:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Task 2: Use KFold for splitting
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Task 3: Train a RandomForest model
model = RandomForestRegressor(random_state=42)

mae_scores = []

for train_index, test_index in kf.split(X):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # Task 4: Evaluate using MAE
    mae = mean_absolute_error(y_test, y_pred)
    mae_scores.append(mae)

# Task 5: Print the averaged score across all folds
print(f"Average MAE across all folds: {sum(mae_scores) / len(mae_scores):.2f}")


In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt
import pandas as pd

feature_importances = pd.DataFrame({'feature': X.columns, 'importance': model.feature_importances_})
feature_importances = feature_importances.sort_values(by='importance', ascending=False)

plt.figure(figsize=(12, 6))
plt.barh(feature_importances['feature'], feature_importances['importance'])
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.title('Feature Importances')
plt.gca().invert_yaxis()
plt.show()


In [ ]:
# Task 2: Write your code here:
import seaborn as sns

y_pred_all = []

# Re-run prediction on full dataset for consistent plotting, or collect all predictions during CV
# For simplicity, let's train on the full dataset and predict to get a single set of predictions
model_full = RandomForestRegressor(random_state=42)
model_full.fit(X, y)
y_pred_full = model_full.predict(X)

plt.figure(figsize=(10, 6))
sns.histplot(y_pred_full, kde=True)
plt.title('Distribution of Predicted Delivery Time')
plt.xlabel('Predicted Delivery Time (minutes)')
plt.ylabel('Frequency')
plt.show()


In [ ]:
# Task Bonus: Write your code here:
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error
import numpy as np

kf = KFold(n_splits=5, shuffle=True, random_state=42)

ensemble_mae_scores = []

for fold, (train_index, test_index) in enumerate(kf.split(X)):
    print(f"\nFold {fold+1}")
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Model 1: RandomForestRegressor
    rf_model = RandomForestRegressor(random_state=42)
    rf_model.fit(X_train, y_train)
    rf_preds = rf_model.predict(X_test)

    # Model 2: CatBoostRegressor
    # verbose=0 to suppress extensive output during training
    cat_model = CatBoostRegressor(random_state=42, verbose=0)
    cat_model.fit(X_train, y_train)
    cat_preds = cat_model.predict(X_test)

    # Average the predictions
    averaged_preds = (rf_preds + cat_preds) / 2

    # Calculate MAE for averaged predictions
    mae = mean_absolute_error(y_test, averaged_preds)
    ensemble_mae_scores.append(mae)
    print(f"MAE for ensemble in Fold {fold+1}: {mae:.4f}")

# Print the averaged MAE across all folds for the ensemble model
print(f"\nAverage MAE for ensemble across all folds: {np.mean(ensemble_mae_scores):.4f}")
